[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/02_ridge_regression/Ridge_Regression.ipynb)  [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/kadimetla/AIML_Learning_Gang/blob/main/statistics/regression_methods/02_ridge_regression/Ridge_Regression.ipynb)

# Notebook 02 — Ridge Regression
## Preventing Overfitting by Penalising Large Weights

**Dataset**: California Housing
**Question**: *What is the median house value — without overfitting to noise?*
**Type**: Supervised Regression with L2 Regularisation

---

## Section 1 — What Is Ridge Regression?

### The problem with plain linear regression

Linear regression finds the weights that minimise prediction error on training data.
With many features, it can assign very large weights — including to features that are
really just capturing noise in the training set.
When applied to new data, those large noisy weights cause big errors. This is **overfitting**.

### The Ridge fix: add a penalty for large weights

Ridge regression modifies the loss function:
```
Ridge Loss = MSE + α × Σ(w²)
```

The second term `α × Σ(w²)` is a penalty that grows whenever any weight gets large.
The model must now balance two goals:
- Fit the training data well (low MSE)
- Keep weights small (low penalty)

> **Result**: weights are shrunk toward zero. Features that mostly capture noise
> get shrunk the most. Predictions on new data become more stable.

### The α hyperparameter

- `α = 0` → no penalty → identical to plain linear regression
- `α = 1` → moderate shrinkage
- `α = 100` → heavy shrinkage → weights near zero → underfitting

The right α is chosen by cross-validation.

## Section 2 — How Does It Learn?

Identical to linear regression gradient descent, but the gradient has an extra term:

```
gradient = −2 × (X^T)(y − Xw) + 2αw
```

The `2αw` term pushes every weight back toward zero at each update.
Features that are genuinely predictive resist this push and keep meaningful weights.
Features that are noise give up their weight under the pressure.

> The L2 penalty is also called **weight decay** in neural network literature —
> the exact same principle keeps neural network weights from growing too large.

## Section 3 — What Does the Data Need?

### Gradient-based models need scaling

All three (Linear, Ridge, Lasso) use gradient descent — the same scaling argument applies:
features on different scales cause unequal gradient steps.

| Feature | Description | Transform |
|---|---|---|
| `MedInc` | Median income (in $10k) | StandardScaler |
| `HouseAge` | Median house age (years) | StandardScaler |
| `AveRooms` | Average rooms per household | StandardScaler + clip outliers |
| `AveBedrms` | Average bedrooms per household | StandardScaler + clip outliers |
| `Population` | Block group population | StandardScaler |
| `AveOccup` | Average household occupancy | StandardScaler + clip outliers |
| `Latitude` | Geographic latitude | StandardScaler |
| `Longitude` | Geographic longitude | StandardScaler |

> **Split first, then scale** — fit the scaler on training data only to avoid data leakage.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110

housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseVal'] = housing.target
for col in ['AveRooms', 'AveBedrms', 'AveOccup']:
    df[col] = df[col].clip(upper=df[col].quantile(0.99))

FEATURES = housing.feature_names.tolist()
TARGET   = 'MedHouseVal'

print(f"California Housing: {df.shape[0]} districts, {len(FEATURES)} features")
print(f"Target (MedHouseVal): median house value in $100k, range {df[TARGET].min():.2f}–{df[TARGET].max():.2f}")
df.head(6)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

X_raw = df[FEATURES].values
y     = df[TARGET].values

X_tr_raw, X_te_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_tr_raw)
X_test  = scaler.transform(X_te_raw)

stats = pd.DataFrame({
    'Feature'     : FEATURES,
    'Raw mean'    : X_tr_raw.mean(axis=0).round(3),
    'Raw std'     : X_tr_raw.std(axis=0).round(3),
    'Scaled mean' : X_train.mean(axis=0).round(4),
    'Scaled std'  : X_train.std(axis=0).round(4),
})
print(f"Train: {len(X_train)} districts  |  Test: {len(X_test)} districts")
print()
print(stats.to_string(index=False))

## Section 4 — Train the Model and Read the Rules

In [ ]:
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# Compare several alpha values
alphas = [0.01, 0.1, 1, 10, 100, 1000]
results = []
for a in alphas:
    m = Ridge(alpha=a)
    m.fit(X_train, y_train)
    y_p = m.predict(X_test)
    results.append({
        'alpha': a,
        'RMSE': np.sqrt(mean_squared_error(y_test, y_p)),
        'R²': r2_score(y_test, y_p),
        'Max |weight|': np.abs(m.coef_).max(),
    })

# Plain linear regression (alpha=0) as baseline
lr = LinearRegression().fit(X_train, y_train)
y_lr = lr.predict(X_test)
print("alpha=0 (linear): "
      f"RMSE={np.sqrt(mean_squared_error(y_test, y_lr)):.4f}  "
      f"R²={r2_score(y_test, y_lr):.4f}  "
      f"Max|w|={np.abs(lr.coef_).max():.4f}")
for r in results:
    print(f"alpha={r['alpha']:6}: RMSE={r['RMSE']:.4f}  R²={r['R²']:.4f}  Max|w|={r['Max |weight|']:.4f}")

In [ ]:
# Coefficient path — how weights change as alpha increases
alphas_path = np.logspace(-2, 4, 200)
coef_paths  = np.array([Ridge(alpha=a).fit(X_train, y_train).coef_ for a in alphas_path])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

for i, feat in enumerate(FEATURES):
    ax1.plot(np.log10(alphas_path), coef_paths[:, i], lw=1.5, label=feat)
ax1.axhline(0, color='black', lw=0.6, ls='--')
ax1.set_xlabel('log10(alpha)  — regularisation strength →', fontsize=11)
ax1.set_ylabel('Feature weight', fontsize=11)
ax1.set_title('Ridge: Coefficient Path\n(weights shrink toward 0 as alpha grows)', fontsize=12)
ax1.legend(fontsize=8, ncol=2)

rmses = [np.sqrt(mean_squared_error(y_test, Ridge(alpha=a).fit(X_train, y_train).predict(X_test)))
         for a in alphas_path]
ax2.plot(np.log10(alphas_path), rmses, 'steelblue', lw=2)
ax2.set_xlabel('log10(alpha)', fontsize=11)
ax2.set_ylabel('Test RMSE', fontsize=11)
ax2.set_title('Test RMSE vs Alpha\n(sweet spot between underfitting and overfitting)', fontsize=12)

plt.tight_layout()
plt.show()
print('Ridge never sets any weight to exactly 0 — it only shrinks them.')

In [ ]:
# Best alpha via cross-validation
from sklearn.linear_model import RidgeCV

ridge_cv = RidgeCV(alphas=np.logspace(-2, 4, 100), cv=5)
ridge_cv.fit(X_train, y_train)
best_alpha = ridge_cv.alpha_
y_pred = ridge_cv.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2   = r2_score(y_test, y_pred)

print(f"Best alpha (5-fold CV): {best_alpha:.4f}")
print(f"RMSE: {rmse:.4f}  |  R²: {r2:.4f}")
print()
coef_df = pd.DataFrame({'Feature': FEATURES, 'Weight': ridge_cv.coef_.round(4)})
coef_df = coef_df.reindex(coef_df['Weight'].abs().sort_values(ascending=False).index)
print(coef_df.to_string(index=False))